In [1]:
# ---- CELL 1 : install + download the code and the data (about 3-5 min) ----------------
import os, subprocess, zipfile
subprocess.run("pip install -q chromadb bitsandbytes accelerate num2words python-multipart fastapi uvicorn "
               "sentence-transformers huggingface_hub", shell=True, check=True)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
REPO = f"{HfApi(token=HF_TOKEN).whoami()['name']}/pathfinder-artifacts"      # the private dataset you created
for fname in ["pathfinder-ai.zip", "pathfinder_artifacts.zip"]:
    path = hf_hub_download(REPO, fname, repo_type="dataset", token=HF_TOKEN)
    zipfile.ZipFile(path).extractall("/kaggle/working" if fname == "pathfinder-ai.zip" else "/kaggle/working/pathfinder-ai")
subprocess.run("ls /kaggle/working/pathfinder-ai /kaggle/working/pathfinder-ai/data", shell=True)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


pathfinder-ai.zip: reconstructing file:   0%|          |  0.00B / 39.6kB            

pathfinder-ai.zip: downloading bytes:           |  0.00B            

pathfinder_artifacts.zip: reconstructing file:   0%|          |  0.00B /  159MB            

pathfinder_artifacts.zip: downloading bytes:           |  0.00B            

/kaggle/working/pathfinder-ai:
app
data
Dockerfile
manifest.json
README.md
requirements.lock.txt
requirements.txt

/kaggle/working/pathfinder-ai/data:
chroma_db
chunks_section.jsonl
jobs_clean.csv


CompletedProcess(args='ls /kaggle/working/pathfinder-ai /kaggle/working/pathfinder-ai/data', returncode=0)

In [2]:

#  ---- CELL 2 : start the AI service + the public tunnel ---------------------------------
import re, secrets, time, urllib.request
os.chdir("/kaggle/working/pathfinder-ai")
# Re-running this cell must not leave an OLD server (with an old key) answering on port 8000:
subprocess.run("pkill -f 'uvicorn app.main'; pkill -f cloudflared", shell=True); time.sleep(3)
try:
    API_KEY = UserSecretsClient().get_secret("API_KEY")
except Exception:
    API_KEY = secrets.token_urlsafe(24)
 
server = subprocess.Popen(["python", "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
                          env={**os.environ, "API_KEY": API_KEY}, stdout=open("server.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
 
CF = "/kaggle/working/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, 0o755)
tunnel = subprocess.Popen([CF, "tunnel", "--url", "http://localhost:8000"], stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
 
URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        URL = m.group(0); break
    time.sleep(1)
print("\nPut these 2 lines in the website's .env.local :\n")
print(f"AI_API_URL={URL}\nAI_API_KEY={API_KEY}\nAI_MOCK=false\n")
 


Put these 2 lines in the website's .env.local :

AI_API_URL=https://apache-illinois-farmer-requesting.trycloudflare.com
AI_API_KEY=pf-7Kx92mQvLa84ZtR3
AI_MOCK=false



In [6]:

# ---- CELL 3 : wait until the models are loaded (first time: 5-10 min) ------------------
import requests
for i in range(180):
    try:
        h = requests.get("http://localhost:8000/health", timeout=3).json()
    except Exception:
        h = {"ok": False, "error": "server starting"}
    if h.get("ok") or (h.get("error") and h["error"] != "server starting"):
        break
    time.sleep(10)
print(h)
if h.get("ok"):                                   # check that the key printed in cell 2 is really the one the server uses
    code = requests.get("http://localhost:8000/result/none", headers={"x-api-key": API_KEY}, timeout=5).status_code
    print("API key check:", "OK" if code == 404 else f"PROBLEM (status {code}): the server uses another key, restart the session")
if not h.get("ok"):
    print("\n--- last lines of server.log ---"); print(open("server.log").read()[-3000:])


{'ok': True, 'model': '7b', 'error': None}
API key check: OK


In [7]:

# ---- CELL 4 : keep the session alive (leave this cell running while you demo) ----------
# Kaggle stops idle sessions; a running cell counts as activity. Stop it (square button) when you are done.
while True:
    try:
        ok = requests.get("http://localhost:8000/health", timeout=5).json().get("ok")
    except Exception:
        ok = False
    gpu = subprocess.run("nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader", shell=True,
                         capture_output=True, text=True).stdout.strip().replace("\n", " | ")
    last = [l for l in open("server.log").read().splitlines() if "/result/" not in l and "/health" not in l][-3:]
    print(time.strftime("%H:%M:%S"), "| AI ok:", ok, "| GPU:", gpu, "|", URL)
    for l in last: print("   log:", l[:160])
    time.sleep(120)

11:18:06 | AI ok: True | GPU: 0 %, 8177 MiB | 0 %, 8277 MiB | https://apache-illinois-farmer-requesting.trycloudflare.com
   log: Loading weights: 100%|██████████| 479/479 [00:04<00:00, 119.32it/s]
   log: [transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. Mor
11:20:06 | AI ok: True | GPU: 0 %, 8177 MiB | 0 %, 8277 MiB | https://apache-illinois-farmer-requesting.trycloudflare.com
   log: Loading weights: 100%|██████████| 479/479 [00:04<00:00, 119.32it/s]
   log: [transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. Mor
11:22:07 | AI ok: True | GPU: 0 %, 8177 MiB | 0 %, 8277 MiB | https://apache-illinois-farmer-requesting.trycloudflare.com
   log: Loading weights: 100%|██████████| 479/479 [00:04<00:00, 119.32it/s]
   log: [transformers] Using `chunk_length_s` is very experimental w

KeyboardInterrupt: 